## Fine Tune `Phi-4-mini-instruct`

In [ ]:
!nvidia-smi

## Sanity Check With Unsloth
`Phi-4-mini-Instruct` fails to work properly with **unsloth** as it generated garabage outputs for inference using unsloth's custom weights. Hence, ignoring that for fine tuning also

In [ ]:
import torch
import json
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="microsoft/Phi-4-mini-instruct",
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=torch.bfloat16
)
# FastLanguageModel.for_inference(model)

sanity = tokenizer.apply_chat_template(
    [{"role": "user", "content": "why is the sky blue?"}],
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    sanity_out = model.generate(
        input_ids=sanity,
        max_new_tokens=256, # keep small for sanity check
        do_sample=False
    )

sanity_response = tokenizer.decode(
    sanity_out[0][sanity.shape[1]:],
    skip_special_tokens=True
).strip()

print(sanity_response)

## Sanity Check With Transformers
The **transformers** framework is a stable **huggingface** framework for fine tuning. It is more mature and stable than unsloth and has support for far more models. Unsloth is great for fine tuning stable, very popular models on minimum hardware. It also fine tunes models with custom weights and kernels which drastically reduce training time and VRAM use.

Since the model was able to perform inference using this framework, it makes sense to use **transformers** for fine tuning also to avoid garbage outputs.  

For fine tuning, _garbage in -> garbage out_ 

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-4-mini-instruct")
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-4-mini-instruct",
    quantization_config=bnb_config,
    device_map="auto",
)

messages = [
    {"role": "system", "content": "end every answer with '-xxx-'"},
    {"role": "user", "content": "why is the sky blue?"}
]

inputs = tokenizer.apply_chat_template(
    messages, 
    tokenize=True,  
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    out = model.generate(
        input_ids=inputs, 
        max_new_tokens=256, 
        do_sample=False
    )

response = tokenizer.decode(
    out[0][inputs.shape[1]:],
    skip_special_tokens=True
).strip()
print(response)

## Inference Testing - Unsloth
Before fine tuning, it is imperative to run inference on at least one randomly selected clean record to ensure the base model at least performs to basic expectations. If the base model fails to generate meaningful output wrt context, it is highly unlikely that fine tuning will now massively improve the base model.

Since sanity check with **unsloth** failed for `Phi-4-mini-Instruct` we can safely assume that the inference test will also fail. 

In [ ]:
import torch
import json
from datasets import load_from_disk
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template


# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-mini-instruct-unsloth-bnb-4bit",
    max_seq_length=4096,
    load_in_4bit=True
)
# Apply chat template to tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template="phi-4"
)

FastLanguageModel.for_inference(model)

print(f"\n✅ model and tokenizer initialized for inference\n")

JSON_SCHEMA = {
    "summary": "A concise, 1-2 sentence abstractive summary of the clinical scenario.",
    "clinical_reasoning": "A step-by-step logical breakdown of the diagnoses, treatments, or clinical decisions made in the text. Explain WHY certain relationships exist. Keep short and brief but to the point.",
    "relationships": [
        {
            "subject": "Source entity (e.g., Patient, Drug, Symptom)",
            "predicate": "Use STANDARD POSITIVE relationships (e.g., HAS_HISTORY, SHOWS_SYMPTOM, DIAGNOSED_WITH, PRESCRIBED). Do not use negated verbs like 'DENIES' or 'LACKS'.",
            "object": "Target entity",
            "polarity": "positive OR negative (Use 'negative' if the patient denies the history or lacks the symptom)",
            "certainty": "confirmed, suspected, OR hedged",
            "evidence": "The exact verbatim text snippet that proves this relationship."
        }
    ],
    "keywords": ["List", "of", "important", "clinical", "NER", "terms"]
}

# Convert schema to string
schema_string = json.dumps(JSON_SCHEMA, indent=4)

records = load_from_disk("/mnt/huggingface/data/medgemma_extracts/test")
record = records.filter(lambda x: x["id"] == "7246274-1")[0]

if record:
    print(f"\n✅ successfully loaded record {record['id']} from disk...\n")
else:
    print(f"\n⚠️ record with id {record['id']} not found!!\n")

raw_medical_text = record["raw_medical_text"]
ground_truth = record["data"]
record_id = record["id"]

# Format prompts
messages = [
    {
        "role": "system",
        "content": f"""
        You are an expert clinical data extractor. Extract data strictly into this JSON schema:
        
        \n\n{schema_string}\n\n

        --------------------------\n
        CRITICAL FORMATTING RULES\n
        --------------------------\n
        1. Use ONLY double quotes for all JSON keys and string values.\n
        2. If clinical terms contain apostrophes (e.g., patient's, Alzheimer's), leave them as raw characters inside the double-quoted strings.\n
        3. Your entire response MUST start strictly with the '{{' character and end strictly with the '}}' character.\n
        4. Output raw JSON only. No explanations, no markdown formatting, no code blocks.\n
        5. Ensure you have values for all the keys specified in the JSON schema.
        \n\n"""
    },
    {
        "role": "user",
        "content": f"""CONTEXT:\n{raw_medical_text}"""
    },
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=2048,
        use_cache=True,
        temperature=0.3,
        do_sample=True,
        min_p=0.1,
        repetition_penalty=1.1
    )

## Inference Testing - Transformers

As the `Phi-4-mini-Instruct` showed successful inference performance, we can safely use this model to generate the medical data extraction as expected from the random test set record. 

In [ ]:
import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-4-mini-instruct")
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-4-mini-instruct",
    quantization_config=bnb_config,
    device_map="auto",
)

JSON_SCHEMA = {
    "summary": "A concise, 1-2 sentence abstractive summary of the clinical scenario.",
    "clinical_reasoning": "A step-by-step logical breakdown of the diagnoses, treatments, or clinical decisions made in the text. Explain WHY certain relationships exist. Keep short and brief but to the point.",
    "relationships": [
        {
            "subject": "Source entity (e.g., Patient, Drug, Symptom)",
            "predicate": "Use STANDARD POSITIVE relationships (e.g., HAS_HISTORY, SHOWS_SYMPTOM, DIAGNOSED_WITH, PRESCRIBED). Do not use negated verbs like 'DENIES' or 'LACKS'.",
            "object": "Target entity",
            "polarity": "positive OR negative (Use 'negative' if the patient denies the history or lacks the symptom)",
            "certainty": "confirmed, suspected, OR hedged",
            "evidence": "The exact verbatim text snippet that proves this relationship."
        }
    ],
    "keywords": ["List", "of", "important", "clinical", "NER", "terms"]
}

# Convert schema to string
schema_string = json.dumps(JSON_SCHEMA, indent=4)
prompt = f"""
You are an expert clinical data extractor. Extract data strictly into this JSON schema:

\n\n{schema_string}\n\n

--------------------------\n
CRITICAL FORMATTING RULES\n
--------------------------\n
1. Use ONLY double quotes for all JSON keys and string values.\n
2. If clinical terms contain apostrophes (e.g., patient's, Alzheimer's), leave them as raw characters inside the double-quoted strings.\n
3. Your entire response MUST start strictly with the '{{' character and end strictly with the '}}' character.\n
4. Output raw JSON only. No explanations, no markdown formatting, no code blocks.\n
5. Ensure you have values for all the keys specified in the JSON schema.\n\n"""

records = load_from_disk("/mnt/huggingface/data/medgemma_extracts/test")
record = records.filter(lambda x: x["id"] == "7246274-1")[0]

if record:
    print(f"\n✅ successfully loaded record {record['id']} from disk...\n")
else:
    print(f"\n⚠️ record with id {record['id']} not found!!\n")

raw_medical_text = record["raw_medical_text"]
ground_truth = record["data"]
record_id = record["id"]

# Format prompts
messages = [
    {
        "role": "system",
        "content": prompt
    },
    {
        "role": "user",
        "content": f"""CONTEXT:\n{raw_medical_text}"""
    },
]

inputs = tokenizer.apply_chat_template(
    messages, 
    tokenize=True,  
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    out = model.generate(
        input_ids=inputs, 
        max_new_tokens=2048, 
        do_sample=False
    )

response = tokenizer.decode(
    out[0][inputs.shape[1]:],
    skip_special_tokens=True
).strip()

print(response)